# 02 · EDA — Best sellers vs. new products (part 2)

**Full extract, no sampling.** DuckDB aggregates all 3.49M rows off disk; pandas only ever
receives the finished result.

Grain: one row per **product** (the extracted name), with earliest sale date, total revenue
and total units.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "to_share" / "inputs.csv").exists()
)
SALES_CSV = ROOT / "to_share" / "inputs.csv"
OUT_CSV = ROOT / "02_product_bestsellers.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

con = duckdb.connect()
con.execute("PRAGMA threads=4")

print(SALES_CSV)

/Users/deepanvishal.thulasivel/Library/CloudStorage/OneDrive-COLORIMAGEINC/Git Local/Product_level_forecast/Product_level_forecast/to_share/inputs.csv


## 1. Base table — `product_id`, extracted `product`, `resolved_color`

Same derivation as the production build (`build_dashboard_data.py`, `sku_product` CTE):
strip the trailing colour off `product_title` to get `product`, and recover the colour from
the title when the `color` column is null.

In [2]:
TITLE = "nullif(trim(CAST(product_title AS VARCHAR)), '')"
COLOR = "nullif(trim(CAST(color AS VARCHAR)), '')"

PRODUCT_SQL = rf"""
    CASE
        WHEN {TITLE} IS NULL THEN concat('Unknown SKU ', sku)
        WHEN {COLOR} IS NOT NULL
             AND ends_with(lower({TITLE}), lower(concat(' - ', {COLOR})))
            THEN rtrim(left({TITLE}, length({TITLE}) - length({COLOR}) - 3))
        WHEN regexp_matches({TITLE}, '\s+-\s+')
            THEN regexp_replace({TITLE}, '\s+-\s+.*$', '')
        ELSE {TITLE}
    END
"""

RESOLVED_COLOR_SQL = rf"""
    coalesce({COLOR}, nullif(regexp_extract({TITLE}, '\s+-\s+(.+)$', 1), ''))
"""

con.execute(f"""
CREATE OR REPLACE VIEW base AS
SELECT
    product_id,
    sku,
    {PRODUCT_SQL}        AS product,
    {COLOR}              AS color,
    {RESOLVED_COLOR_SQL} AS resolved_color,
    nullif(trim(CAST(item_class  AS VARCHAR)), '') AS item_class,
    nullif(trim(CAST(subcategory AS VARCHAR)), '') AS subcategory,
    nullif(trim(CAST(category    AS VARCHAR)), '') AS category,
    order_date,
    revenue,
    ordered_quantities   AS units
FROM read_csv_auto('{SALES_CSV}', sample_size=200000)
""")

print(con.execute("SELECT count(*) FROM base").fetchone()[0], "rows")
con.execute("SELECT * FROM base LIMIT 8").df()

3485426 rows


,product_id,sku,product,color,resolved_color,item_class,subcategory,category,order_date,revenue,units
0,4594792136822,A0029U016641,Grounded No-Slip Towel,Jungle,Jungle,Towels,Towels,Equipment,2025-01-01,340.00,5
1,<NA>,A0029U011,Grounded No-Slip Towel,None,Black,None,None,None,2025-01-01,340.00,5
2,4579650633846,A0082W025124,Airbrush Invisible Thong,Smoky Quartz,Smoky Quartz,Underwear,Undergarments,Accessories,2025-01-01,15.30,1
3,<NA>,A0059U001,Glow Wristband (2-Pack),None,White,None,None,None,2025-01-01,87.97,3
4,<NA>,A0084U011,Uplifting Yoga Block,None,Black,None,None,None,2025-01-01,191.66,7
5,<NA>,A0082W012,Airbrush Invisible Thong,None,Black,None,None,None,2025-01-01,54.13,3
6,8833308328217,A0094W012,Women's Pivot Barre Sock,Black,Black,Socks,Socks,Accessories,2025-01-01,544.84,20
7,4594792497270,A0084U040061,Uplifting Yoga Block,Jungle,Jungle,Blocks,Blocks,Equipment,2025-01-01,84.00,3


## 2. Group by product

`first_sale_date` is what separates a genuine best seller from a product that simply
launched early — a high total on a recent first sale date is a much stronger signal.

In [3]:
best_sellers = con.execute("""
    SELECT
        product,
        min(order_date) AS first_sale_date,
        max(order_date) AS last_sale_date,
        sum(revenue)    AS total_sales,
        sum(units)      AS total_units
    FROM base
    GROUP BY product
    ORDER BY total_sales DESC
""").df()

best_sellers.to_csv(OUT_CSV, index=False)

print(f"{len(best_sellers):,} products  ->  {OUT_CSV.name}")
print(f"total sales : ${best_sellers['total_sales'].sum():,.2f}")
print(f"total units : {best_sellers['total_units'].sum():,}")
best_sellers.head(25)

1,981 products  ->  02_product_bestsellers.csv
total sales : $2,831,081,580.32
total units : 31,761,946.0


,product,first_sale_date,last_sale_date,total_sales,total_units
0,Accolade Crew Neck Pullover,2025-01-01,2026-09-15,1.798396e+08,1596399.0
1,Accolade Straight Leg Sweatpant,2025-01-01,2026-09-15,1.252525e+08,1132577.0
2,Suit Up Trouser (Regular),2025-01-01,2026-09-15,1.174930e+08,998611.0
3,Accolade 1/4 Zip Pullover,2025-01-01,2026-09-15,1.132722e+08,962129.0
4,Accolade Sweatpant,2025-01-01,2026-09-15,6.422549e+07,582402.0
5,Accolade Hoodie,2025-01-01,2026-09-15,6.324453e+07,525401.0
6,7/8 High-Waist Airlift Legging,2025-01-01,2026-09-15,6.068876e+07,565407.0
7,Match Point Short,2025-01-01,2026-09-15,4.150347e+07,654455.0
8,Cropped Accolade Crewneck,2025-01-01,2026-09-15,3.834750e+07,354477.0
9,Accolade Full Zip Hoodie,2025-01-01,2026-09-15,3.621770e+07,304790.0


## 3. Group by raw `color`

The `color` column exactly as it arrives in the extract — no title fallback. Rows where the
source has no colour group into a single `NULL` row, so the totals still reconcile.

In [4]:
color_summary = con.execute("""
    SELECT
        color,
        min(order_date) AS first_sale_date,
        max(order_date) AS last_sale_date,
        sum(revenue)    AS total_sales,
        sum(units)      AS total_units
    FROM base
    GROUP BY color
    ORDER BY total_sales DESC
""").df()

color_summary.to_csv(ROOT / "02_color_summary.csv", index=False)

print(f"{len(color_summary):,} distinct raw colours (including NULL)")
print(f"total sales : ${color_summary['total_sales'].sum():,.2f}")
color_summary.head(20)

585 distinct raw colours (including NULL)
total sales : $2,831,081,580.32


,color,first_sale_date,last_sale_date,total_sales,total_units
0,Black,2025-01-01,2026-09-15,9.415159e+08,10480614.0
1,Navy,2025-01-01,2026-09-15,2.315506e+08,2464170.0
2,Espresso,2025-01-01,2026-09-15,1.607623e+08,1636270.0
3,White,2025-01-01,2026-09-15,1.507788e+08,2520138.0
4,Ivory,2025-01-01,2026-09-15,1.048827e+08,962091.0
5,Gravel,2025-01-01,2026-09-15,8.385054e+07,834052.0
6,None,2025-01-01,2026-09-14,8.314083e+07,949812.0
7,Athletic Grey,2025-01-01,2025-12-14,8.070286e+07,891999.0
8,Athletic Heather Grey,2025-12-15,2026-09-15,5.216975e+07,502622.0
9,Bone,2025-01-01,2026-09-15,4.278172e+07,485118.0


## 4. Group by `resolved_color`

`color` when present, otherwise the text pulled off the end of `product_title`. This is what
the production build keys product identity on.

In [5]:
resolved_color_summary = con.execute("""
    SELECT
        resolved_color,
        min(order_date) AS first_sale_date,
        max(order_date) AS last_sale_date,
        sum(revenue)    AS total_sales,
        sum(units)      AS total_units
    FROM base
    GROUP BY resolved_color
    ORDER BY total_sales DESC
""").df()

resolved_color_summary.to_csv(ROOT / "02_resolved_color_summary.csv", index=False)

print(f"{len(resolved_color_summary):,} distinct resolved colours")
print(f"total sales : ${resolved_color_summary['total_sales'].sum():,.2f}")
resolved_color_summary.head(20)

625 distinct resolved colours
total sales : $2,831,081,580.32


,resolved_color,first_sale_date,last_sale_date,total_sales,total_units
0,Black,2025-01-01,2026-09-15,9.493867e+08,10554085.0
1,Navy,2025-01-01,2026-09-15,2.326237e+08,2474010.0
2,Espresso,2025-01-01,2026-09-15,1.614614e+08,1642341.0
3,White,2025-01-01,2026-09-15,1.526103e+08,2542823.0
4,Ivory,2025-01-01,2026-09-15,1.073621e+08,1000668.0
5,Gravel,2025-01-01,2026-09-15,8.458053e+07,842811.0
6,Athletic Grey,2025-01-01,2025-12-14,8.070286e+07,891999.0
7,Athletic Heather Grey,2025-01-01,2026-09-15,6.068564e+07,584424.0
8,Bone,2025-01-01,2026-09-15,4.320894e+07,490394.0
9,Winter Frost,2025-10-21,2026-09-15,4.061863e+07,423960.0


## 5. `color` vs `resolved_color` — where they disagree

Every `(color, resolved_color)` pair that is **not** identical, i.e. every row where the
fallback changed the answer. Two causes show up here:

- `color` is NULL and the value was recovered from the title (`filled_from_title`)
- both exist but differ — the title suffix and the `color` column use different vocabularies
  (`disagrees`)

In [6]:
color_diff = con.execute("""
    SELECT
        color,
        resolved_color,
        CASE WHEN color IS NULL THEN 'filled_from_title' ELSE 'disagrees' END AS reason,
        count(*)        AS rows,
        min(order_date) AS first_sale_date,
        max(order_date) AS last_sale_date,
        sum(revenue)    AS total_sales,
        sum(units)      AS total_units
    FROM base
    WHERE color IS DISTINCT FROM resolved_color
    GROUP BY color, resolved_color, reason
    ORDER BY total_sales DESC
""").df()

color_diff.to_csv(ROOT / "02_color_vs_resolved_color.csv", index=False)

print(f"{len(color_diff):,} mismatched pairs")
print(color_diff.groupby("reason")[["rows", "total_sales"]].sum().to_string())
color_diff.head(20)

393 mismatched pairs
                    rows  total_sales
reason                               
filled_from_title  83882  79458383.22


,color,resolved_color,reason,rows,first_sale_date,last_sale_date,total_sales,total_units
0,None,Burgundy,filled_from_title,2512,2025-07-31,2025-10-04,9347091.58,82086.0
1,None,Athletic Heather Grey,filled_from_title,3124,2025-01-01,2026-09-14,8515890.15,81802.0
2,None,Black,filled_from_title,10885,2025-01-01,2026-09-14,7870760.57,73471.0
3,None,Candy Heart Pink,filled_from_title,730,2026-01-08,2026-02-08,3022511.52,28574.0
4,None,Macadamia,filled_from_title,2291,2025-03-03,2025-06-08,2580937.07,27149.0
5,None,Ivory,filled_from_title,3283,2025-01-01,2026-09-14,2479476.10,38577.0
6,None,Spearmint,filled_from_title,903,2025-05-20,2025-06-08,2139115.66,24180.0
7,None,Dune Grass,filled_from_title,1292,2026-05-28,2026-08-11,2074280.84,20085.0
8,None,Neon Bubblegum,filled_from_title,349,2025-06-30,2025-07-14,1889003.85,23614.0
9,None,White,filled_from_title,2947,2025-01-01,2026-08-31,1831449.67,22685.0


## 6. Multi-colour values in `color` — primary / secondary split

**Raw `color` column only** — `resolved_color` is not used here.

`/` is the only separator present in the data (no commas, ampersands or "and"). Of 584
distinct non-null colours, **227 are multi-colour**: 207 with two parts, 20 with three.

- `primary_color` — the first part, e.g. `Light Provence Blue` from `Light Provence Blue/White`
- `secondary_colors` — everything after the first `/`, kept whole so three-part values keep
  both, e.g. `White/Grey`

In [7]:
multi_color = con.execute("""
    SELECT
        color,
        trim(split_part(color, '/', 1))                    AS primary_color,
        trim(substr(color, position('/' IN color) + 1))    AS secondary_colors,
        length(color) - length(replace(color, '/', '')) + 1 AS n_colors,
        min(order_date) AS first_sale_date,
        max(order_date) AS last_sale_date,
        sum(revenue)    AS total_sales,
        sum(units)      AS total_units
    FROM base
    WHERE color IS NOT NULL AND color LIKE '%/%'
    GROUP BY color, primary_color, secondary_colors, n_colors
    ORDER BY total_sales DESC
""").df()

multi_color.to_csv(ROOT / "02_multi_color_split.csv", index=False)

print(f"{len(multi_color):,} multi-colour values")
print(f"total sales : ${multi_color['total_sales'].sum():,.2f} "
      f"({multi_color['total_sales'].sum() / 2831081580.32:.1%} of all sales)")
print()
print(multi_color["n_colors"].value_counts().sort_index().to_string())
multi_color.head(20)

227 multi-colour values
total sales : $170,235,728.27 (6.0% of all sales)

n_colors
2    207
3     20


,color,primary_color,secondary_colors,n_colors,first_sale_date,last_sale_date,total_sales,total_units
0,Black/White,Black,White,2,2025-12-15,2026-09-15,38646672.63,483208.0
1,White/Black,White,Black,2,2025-12-15,2026-09-15,13719816.76,249719.0
2,Black/Black,Black,Black,2,2025-12-15,2026-09-15,11213791.33,127178.0
3,Candy Heart Pink/White,Candy Heart Pink,White,2,2026-01-08,2026-09-15,8377781.37,94521.0
4,Navy/White,Navy,White,2,2025-12-15,2026-09-15,7476432.21,104716.0
5,Light Provence Blue/White,Light Provence Blue,White,2,2026-04-13,2026-09-15,6851856.66,78342.0
6,Athletic Heather Grey/White,Athletic Heather Grey,White,2,2025-12-15,2026-09-15,6824969.76,80355.0
7,White/White,White,White,2,2025-12-15,2026-09-15,6448911.23,52352.0
8,Black/Ivory,Black,Ivory,2,2025-12-15,2026-09-15,4818650.28,39538.0
9,Black/Titanium,Black,Titanium,2,2025-12-15,2026-09-15,4031748.06,49501.0


## 7. Summary by `primary_color`

The multi-colour population from section 6, rolled up to the **primary colour only** —
secondary colours are collapsed away.

In [8]:
primary_color_summary = con.execute("""
    SELECT
        trim(split_part(color, '/', 1)) AS primary_color,
        sum(revenue) AS total_sales,
        sum(units)   AS total_units
    FROM base
    WHERE color IS NOT NULL AND color LIKE '%/%'
    GROUP BY primary_color
    ORDER BY total_sales DESC
""").df()

primary_color_summary.to_csv(ROOT / "02_primary_color_summary.csv", index=False)

print(f"{len(primary_color_summary):,} primary colours "
      f"(from 227 multi-colour values)")
print(f"total sales : ${primary_color_summary['total_sales'].sum():,.2f}")
primary_color_summary.head(20)

107 primary colours (from 227 multi-colour values)
total sales : $170,235,728.27


,primary_color,total_sales,total_units
0,Black,59630802.92,709175.0
1,White,26055653.01,487440.0
2,Navy,12770319.35,162837.0
3,Candy Heart Pink,8576528.66,98303.0
4,Light Provence Blue,6912578.13,80668.0
5,Athletic Heather Grey,6825267.80,80365.0
6,Bright Red,3915917.06,44421.0
7,Vintage Pink,3307873.87,43400.0
8,Azure Blue,2814319.35,32179.0
9,Oatmeal Heather,2785010.70,37635.0


## 8. Group by `item_class`

Taxonomy rollup — the narrowest level — individual garment types. Rows with no value group into a single `NULL` row so the totals
still reconcile to the full extract.

In [9]:
item_class_summary = con.execute("""
    SELECT
        item_class,
        sum(revenue) AS total_sales,
        sum(units)   AS total_units
    FROM base
    GROUP BY item_class
    ORDER BY total_sales DESC
""").df()

item_class_summary.to_csv(ROOT / "02_item_class_summary.csv", index=False)

print(f"{len(item_class_summary):,} distinct item_class values (including NULL)")
print(f"total sales : ${item_class_summary['total_sales'].sum():,.2f}")
item_class_summary.head(20)

77 distinct item_class values (including NULL)
total sales : $2,831,081,580.32


,item_class,total_sales,total_units
0,Pullovers,4.302444e+08,3854582.0
1,Leggings,2.920606e+08,2740116.0
2,Sweatpants,2.655856e+08,2438829.0
3,Pants,2.605375e+08,2151066.0
4,Jackets,2.032009e+08,1249857.0
5,Hoodies,1.827301e+08,1561910.0
6,Shorts,1.797222e+08,2746053.0
7,Bras,1.536601e+08,2349973.0
8,Tanks,1.258988e+08,2089090.0
9,Long Sleeves,1.093913e+08,1345151.0


## 9. Group by `subcategory`

Taxonomy rollup — the mid level, above item class. Rows with no value group into a single `NULL` row so the totals
still reconcile to the full extract.

In [10]:
subcategory_summary = con.execute("""
    SELECT
        subcategory,
        sum(revenue) AS total_sales,
        sum(units)   AS total_units
    FROM base
    GROUP BY subcategory
    ORDER BY total_sales DESC
""").df()

subcategory_summary.to_csv(ROOT / "02_subcategory_summary.csv", index=False)

print(f"{len(subcategory_summary):,} distinct subcategory values (including NULL)")
print(f"total sales : ${subcategory_summary['total_sales'].sum():,.2f}")
subcategory_summary.head(20)

71 distinct subcategory values (including NULL)
total sales : $2,831,081,580.32


,subcategory,total_sales,total_units
0,Coverups,6.404210e+08,5597877.0
1,Leggings,2.920606e+08,2740116.0
2,Sweatpants,2.655856e+08,2438829.0
3,Pants,2.605375e+08,2151066.0
4,Jackets,2.032009e+08,1249857.0
5,Shorts,1.797222e+08,2746053.0
6,Bras,1.536601e+08,2349973.0
7,Tanks,1.258988e+08,2089090.0
8,Long Sleeves,1.093913e+08,1345151.0
9,Short Sleeves,1.032478e+08,1682112.0


## 10. Group by `category`

Taxonomy rollup — the broadest of the three. Rows with no value group into a single `NULL` row so the totals
still reconcile to the full extract.

In [11]:
category_summary = con.execute("""
    SELECT
        category,
        sum(revenue) AS total_sales,
        sum(units)   AS total_units
    FROM base
    GROUP BY category
    ORDER BY total_sales DESC
""").df()

category_summary.to_csv(ROOT / "02_category_summary.csv", index=False)

print(f"{len(category_summary):,} distinct category values (including NULL)")
print(f"total sales : ${category_summary['total_sales'].sum():,.2f}")
category_summary.head(20)

29 distinct category values (including NULL)
total sales : $2,831,081,580.32


,category,total_sales,total_units
0,Bottoms,1.076066e+09,11175408.0
1,Outerwear,8.532978e+08,6920721.0
2,Tops,3.385381e+08,5116356.0
3,Bras,1.536601e+08,2349973.0
4,Shoes,1.066990e+08,654206.0
5,None,6.793621e+07,702887.0
6,One Piece,5.686714e+07,495489.0
7,Accessories,5.298746e+07,1414986.0
8,Hats,5.069668e+07,884402.0
9,Socks,4.097084e+07,1509413.0
